# Twitter Stock Data Exploration

Load the stock tweets dataset, preview rows, and review the inferred schema.

In [5]:
import pandas as pd
from pathlib import Path

data_path = Path(r"D:\\Projects\\school\\Data-analysis\\stock_tweets.csv")
if not data_path.exists():
    data_path = Path("..") / "stock_tweets.csv"
df = pd.read_csv(data_path)
df.head()

,Date,Tweet,Stock Name,Company Name
0,2022-09-29 23:41:16+00:00,Mainstream media has done an amazing job at br...,TSLA,"Tesla, Inc."
1,2022-09-29 23:24:43+00:00,Tesla delivery estimates are at around 364k fr...,TSLA,"Tesla, Inc."
2,2022-09-29 23:18:08+00:00,3/ Even if I include 63.0M unvested RSUs as of...,TSLA,"Tesla, Inc."
3,2022-09-29 22:40:07+00:00,@RealDanODowd @WholeMarsBlog @Tesla Hahaha why...,TSLA,"Tesla, Inc."
4,2022-09-29 22:27:05+00:00,"@RealDanODowd @Tesla Stop trying to kill kids,...",TSLA,"Tesla, Inc."


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 80793 entries, 0 to 80792
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Date          80793 non-null  str  
 1   Tweet         80793 non-null  str  
 2   Stock Name    80793 non-null  str  
 3   Company Name  80793 non-null  str  
dtypes: str(4)
memory usage: 2.5 MB


In [7]:
schema = df.dtypes
schema

Date            str
Tweet           str
Stock Name      str
Company Name    str
dtype: object

In [8]:
df.describe(include="all").T

,count,unique,top,freq
Date,80793,64424,2022-07-07 18:32:41+00:00,14
Tweet,80793,64479,$TSLA will triple in 2022 🚀🌕,25
Stock Name,80793,25,TSLA,37422
Company Name,80793,25,"Tesla, Inc.",37422


## Sentiment analysis

Compute TextBlob and VADER scores, plus a combined sentiment label.

In [9]:
import re
from collections import Counter

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob
import plotly.express as px


def ensure_nltk_data():
    resources = [
        ("sentiment/vader_lexicon.zip", "vader_lexicon"),
        ("tokenizers/punkt.zip", "punkt"),
    ]
    for resource_path, resource_name in resources:
        try:
            nltk.data.find(resource_path)
        except LookupError:
            nltk.download(resource_name)


ensure_nltk_data()

In [10]:
TOKEN_PATTERN = re.compile(r"[A-Za-z]{2,}")


def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"@[A-Za-z0-9_]+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"RT[\s]+", "", text)
    text = re.sub(r"https?://\S+", "", text)
    return text.strip()


def get_final_sentiment(combined_score):
    if combined_score >= 0.05:
        return "Positive"
    if combined_score <= -0.05:
        return "Negative"
    return "Neutral"

In [11]:
tweet_col = None
for col in ["tweet", "text", "Tweet", "Text", "full_text"]:
    if col in df.columns:
        tweet_col = col
        break

if tweet_col is None:
    raise ValueError("Could not find a tweet text column. Expected one of: tweet, text, full_text")

sentiment_df = df.copy()
sentiment_df["raw_tweet"] = sentiment_df[tweet_col].astype(str)
sentiment_df["cleaned_tweet"] = sentiment_df["raw_tweet"].apply(clean_text)

sentiment_df["textblob_polarity"] = sentiment_df["cleaned_tweet"].apply(
    lambda text: TextBlob(text).sentiment.polarity
)
sentiment_df["textblob_subjectivity"] = sentiment_df["cleaned_tweet"].apply(
    lambda text: TextBlob(text).sentiment.subjectivity
)

sia = SentimentIntensityAnalyzer()
sentiment_df["vader_compound"] = sentiment_df["cleaned_tweet"].apply(
    lambda text: sia.polarity_scores(text)["compound"]
)

sentiment_df["combined_sentiment_score"] = (
    sentiment_df["textblob_polarity"] + sentiment_df["vader_compound"]
) / 2
sentiment_df["final_sentiment"] = sentiment_df["combined_sentiment_score"].apply(get_final_sentiment)

sentiment_df.head()

,Date,Tweet,Stock Name,Company Name,raw_tweet,cleaned_tweet,textblob_polarity,textblob_subjectivity,vader_compound,combined_sentiment_score,final_sentiment
0,2022-09-29 23:41:16+00:00,Mainstream media has done an amazing job at br...,TSLA,"Tesla, Inc.",Mainstream media has done an amazing job at br...,Mainstream media has done an amazing job at br...,0.600000,0.900000,0.0772,0.338600,Positive
1,2022-09-29 23:24:43+00:00,Tesla delivery estimates are at around 364k fr...,TSLA,"Tesla, Inc.",Tesla delivery estimates are at around 364k fr...,Tesla delivery estimates are at around 364k fr...,0.000000,0.000000,0.0000,0.000000,Neutral
2,2022-09-29 23:18:08+00:00,3/ Even if I include 63.0M unvested RSUs as of...,TSLA,"Tesla, Inc.",3/ Even if I include 63.0M unvested RSUs as of...,3/ Even if I include 63.0M unvested RSUs as of...,0.018182,0.277273,0.2960,0.157091,Positive
3,2022-09-29 22:40:07+00:00,@RealDanODowd @WholeMarsBlog @Tesla Hahaha why...,TSLA,"Tesla, Inc.",@RealDanODowd @WholeMarsBlog @Tesla Hahaha why...,Hahaha why are you still trying to stop Tesla ...,0.079167,0.433333,-0.7568,-0.338817,Negative
4,2022-09-29 22:27:05+00:00,"@RealDanODowd @Tesla Stop trying to kill kids,...",TSLA,"Tesla, Inc.","@RealDanODowd @Tesla Stop trying to kill kids,...","Stop trying to kill kids, you sad deranged old...",-0.200000,0.600000,-0.8750,-0.537500,Negative


In [12]:
sentiment_counts = (
    sentiment_df["final_sentiment"].value_counts().reindex(["Positive", "Neutral", "Negative"]).fillna(0)
)

kpis = {
    "total_tweets": len(sentiment_df),
    "avg_polarity": sentiment_df["textblob_polarity"].mean(),
    "avg_subjectivity": sentiment_df["textblob_subjectivity"].mean(),
    "positive_share": (sentiment_df["final_sentiment"] == "Positive").mean(),
    "negative_share": (sentiment_df["final_sentiment"] == "Negative").mean(),
}

kpis

{'total_tweets': 80793,
 'avg_polarity': np.float64(0.10931177689463487),
 'avg_subjectivity': np.float64(0.36037188260551384),
 'positive_share': np.float64(0.5457527261025089),
 'negative_share': np.float64(0.2023318851880732)}

In [ ]:
sentiment_chart = px.bar(
    sentiment_counts.reset_index(),
    x="final_sentiment",
    y="count",
    title="Sentiment Distribution",
    labels={"final_sentiment": "sentiment", "count": "count"},
    color="final_sentiment",
    color_discrete_map={"Positive": "#2E8B57", "Neutral": "#6C757D", "Negative": "#C0392B"},
)

score_hist = px.histogram(
    sentiment_df,
    x="combined_sentiment_score",
    nbins=30,
    title="Combined Sentiment Score",
)

scatter_fig = px.scatter(
    sentiment_df,
    x="textblob_polarity",
    y="textblob_subjectivity",
    color="final_sentiment",
    title="Polarity vs Subjectivity",
    color_discrete_map={"Positive": "#2E8B57", "Neutral": "#6C757D", "Negative": "#C0392B"},
)

counter = Counter()
for text in sentiment_df["cleaned_tweet"].dropna().astype(str):
    tokens = [token.lower() for token in TOKEN_PATTERN.findall(text)]
    counter.update(tokens)

terms_df = pd.DataFrame(counter.most_common(12), columns=["term", "count"])
terms_fig = px.bar(
    terms_df,
    x="count",
    y="term",
    orientation="h",
    title="Top Terms",
)

sentiment_chart
score_hist
scatter_fig
terms_fig

ValueError: Value of 'x' is not the name of a column in 'data_frame'. Expected one of ['final_sentiment', 'count'] but received: index
 To use the index, pass it in directly as `df.index`.